In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib_inline as mlt
import pickle
import os
import ast

In [3]:
X_train_title = np.load('processed_data/X_train_title.npy')
X_train_text = np.load('processed_data/X_train_text.npy')
X_train_other_features = np.load('processed_data/X_train_other.npy')

X_test_title = np.load('processed_data/X_test_title.npy')
X_test_text = np.load('processed_data/X_test_text.npy')
X_test_other_features = np.load('processed_data/X_test_other.npy')

y_train_ready = np.load('processed_data/y_train.npy')
y_test_ready = np.load('processed_data/y_test.npy')

In [5]:
VOCAB_SIZE = 25000
TITLE_MAX_LENGTH = X_train_title.shape[1]
TEXT_MAX_LENGTH = X_train_text.shape[1] 
EMBEDDING_DIM = 30
NUM_OTHER_FEATURES = X_train_other_features.shape[1]

In [6]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Input, concatenate
from sklearn.metrics import classification_report, confusion_matrix

2025-10-16 17:35:51.391782: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
input_title = Input(shape=(TITLE_MAX_LENGTH,), name='title_input')
input_text = Input(shape=(TEXT_MAX_LENGTH,), name='text_input')
input_other = Input(shape=(NUM_OTHER_FEATURES,), name='other_features_input')

In [8]:
shared_embedding = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM)

In [9]:
embedded_title = shared_embedding(input_title)
gru_title = GRU(64)(embedded_title)

embedded_text = shared_embedding(input_text)
gru_text = GRU(128)(embedded_text)

In [10]:
concatenated = concatenate([gru_title, gru_text,input_other])

In [11]:
dense1 = Dense(128, activation='relu')(concatenated)
dropout = Dropout(0.5)(dense1)
output = Dense(1, activation='sigmoid')(dropout)

In [12]:
model = Model(inputs=[input_title, input_text, input_other], outputs=output)

In [13]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [14]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping_callback = EarlyStopping(
    monitor='val_loss', 
    patience=7,         
    verbose=1,           
    restore_best_weights=True 
)

In [15]:
EPOCHS = 50
BATCH_SIZE = 64
history = model.fit(
    [X_train_title, X_train_text,X_train_other_features], 
    y_train_ready,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=([X_test_title, X_test_text,X_test_other_features], y_test_ready),
    verbose=1,
    callbacks=[early_stopping_callback]
)

Epoch 1/50


2025-10-16 17:36:03.681101: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 215508000 exceeds 10% of free system memory.


301/562 ━━━━━━━━━━━━━━━━━━━━ 8:55 2s/step - accuracy: 0.8042 - loss: 0.4872

KeyboardInterrupt: 

In [18]:
with open('Model_Gru.keras','wb') as file:
    pickle.dump(model,file)